# Active Learning: Coupled Double-Well System

P particles in coupled 1D double-well potentials, evolved with overdamped Langevin dynamics. The noise intensity $\sigma$ controls a sharp unimodal-to-bimodal phase transition.

Acquisition strategies are compared by how efficiently they concentrate labels near the transition regime.

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # replace with desired GPU
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import sys
import jax
import jax.numpy as jnp
import jax.random as jr
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

repo_root = Path("../..").resolve()
example_dir = Path(".").resolve()
for p in [str(repo_root), str(example_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

%load_ext autoreload
%autoreload 2

print("JAX devices:", jax.devices())

In [ ]:
from coupled_double_well import verify_phase_diagram

from active_learning_utils import (
    get_default_mdn_config,
    create_shared_double_well_data,
    make_mdn_trainer_factory,
    run_pool_based_active_learning_experiment,
    plot_loss_comparison,
    plot_sigma_concentration,
    plot_single_run,
)

---
## Step 1 — Verify the Phase Transition

Before running AL, confirm that the simulator produces the expected unimodal-to-bimodal transition as $\sigma$ increases.

`verify_phase_diagram()` plots histograms of the final position of a **single particle** (P=1, no coupling), all starting from $q_0 = -0.5$, for $\sigma = 0.3, 0.7, 1.0, 1.5$.

**What to look for:** At $\sigma=0.3$ you should see a single sharp peak near $q \approx -1$ (particle stayed trapped in the left well). As $\sigma$ increases, a second peak near $q \approx +1$ should appear and grow (particle escapes to the right well via Kramers' mechanism). This confirms the simulator produces the expected unimodal $\to$ bimodal phase transition.

In [ ]:
verify_phase_diagram()

---
## Step 2 — Create the Shared Benchmark Dataset

Generate the pool + test set used by every acquisition strategy below. All methods draw from the same 50,000-point unlabelled pool and are evaluated on the same held-out 2,000 points.

In [ ]:
# ----- System parameters -----
P = 5           # number of particles
T = 5.0         # integration time
DT = 0.005      # time step
N_SNAPSHOTS = 4
SIGMA_RANGE = (0.3, 2.0)
KAPPA_RANGE = (0.0, 3.0)

data = create_shared_double_well_data(
    seed=0,
    candidate_sample_count=50_000,
    test_sample_count=2_000,
    initial_sample_count=100,
    P=P, T=T, dt=DT,
    n_snapshots=N_SNAPSHOTS,
    sigma_range=SIGMA_RANGE,
    kappa_range=KAPPA_RANGE,
)

print(f"Initial labelled : {data['initial_labeled_inputs'].shape[0]}")
print(f"Unlabelled pool  : {data['remaining_pool_inputs'].shape[0]}")
print(f"Test set         : {data['test_data'][0].shape[0]}")
print(f"Input dim        : {data['initial_labeled_inputs'].shape[1]}")
print(f"Output dim       : {data['initial_labeled_targets'].shape[1]}")

---
## Step 3 — Explore the Dataset

Visualise the output structure using a 3,000-sample slice of the pool. The scatter plot below shows particle 0 vs particle 1 at the final snapshot, coloured by $\sigma$.

**What to look for:** At low $\sigma$ (blue/cool), points cluster tightly near $(\pm 1, \pm 1)$ — particles are stuck where they started. At high $\sigma$ (red/warm), points spread across all four quadrants as both particles hop freely between wells. The coupling $\kappa$ may cause diagonal correlations (particles tend to jump together).

In [ ]:
# Scatter: particle 0 vs particle 1 final positions, coloured by sigma
n_show = 3_000
x_slice = data["remaining_pool_inputs"][:n_show]
y_slice = data["remaining_pool_targets"][:n_show]

fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(
    np.asarray(y_slice[:, 0]), np.asarray(y_slice[:, 1]),
    c=np.asarray(x_slice[:, P]), s=3, alpha=0.5, cmap="coolwarm",
)
fig.colorbar(sc, ax=ax, label="$\\sigma$")
ax.set_xlabel("$q_0(T)$")
ax.set_ylabel("$q_1(T)$")
ax.set_title("Output scatter coloured by noise intensity")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 4 — Create the MDN Trainer

In [ ]:
# ----- MDN + ensemble configuration -----
ENSEMBLE_SIZE = 8
N_MIXTURES = 8      # MDN components (enough for correlated particle modes)
HIDDEN_FEATURES = 128
DEPTH = 3
N_ITER = 10_000     # max iterations (cap for adaptive scaling)
BATCH_SIZE = 128
ACQUISITION_BATCH = 256
OUT_DIM = P if N_SNAPSHOTS == 0 else N_SNAPSHOTS * P

# ----- Active learning budget -----
AL_ITERS = 20
QUERY_BATCH_SIZE = 50
# final labeled = 100 + 20*50 = 1,100 out of 50,000 pool (2.2%)

# ----- Adaptive iteration scaling -----
ADAPTIVE_ITERS = True
ITER_PER_SAMPLE = 10

config = get_default_mdn_config(
    out_dim=OUT_DIM,
    hidden_features=HIDDEN_FEATURES,
    depth=DEPTH,
    num_mixtures=N_MIXTURES,
    ensemble_size=ENSEMBLE_SIZE,
)

trainer = make_mdn_trainer_factory(
    config,
    n_iter=N_ITER,
    batch_size=BATCH_SIZE,
    adaptive_iters=ADAPTIVE_ITERS,
    iter_per_sample=ITER_PER_SAMPLE,
)

print("Trainer factory created.")
print(f"  Ensemble size : {ENSEMBLE_SIZE}")
print(f"  MDN mixtures  : {N_MIXTURES}")
print(f"  Network       : {DEPTH} x {HIDDEN_FEATURES}")
print(f"  Output dim    : {OUT_DIM}")
print(f"  Max iters     : {N_ITER}")
print(f"  Adaptive iters: {ADAPTIVE_ITERS}  ({ITER_PER_SAMPLE} iter/sample)")
print(f"  AL rounds     : {AL_ITERS} x batch {QUERY_BATCH_SIZE}")
print(f"  Final labeled : {100 + AL_ITERS * QUERY_BATCH_SIZE}")
print()
print("Iteration schedule (first few rounds):")
for i in range(6):
    n = 100 + i * QUERY_BATCH_SIZE
    eff = min(N_ITER, ITER_PER_SAMPLE * n)
    print(f"  Round {i}: {n:>5d} labeled -> {eff:>6d} iters")

---
## Step 5 — Run Active Learning

Run the full strategy set with the same labelling budget and MDN architecture.

In [ ]:
STRATEGIES = [
    "random",
    "mdn_epistemic_variance",
    "sbal_mdn_epistemic_variance",
    "mi_lb",
    "sbal_mi_lb",
    "bait",
    "coreset",
]

SBAL_TEMPERATURE = 0.3
CORESET_ENSEMBLE_MEMBER = 0
CORESET_POOL_SUBSAMPLE = None

results = {}
for acquisition_name in STRATEGIES:
    kwargs = {}
    if acquisition_name.startswith("sbal_"):
        kwargs["sbal_temperature"] = SBAL_TEMPERATURE
    if acquisition_name == "coreset":
        kwargs.update(
            coreset_ensemble_member=CORESET_ENSEMBLE_MEMBER,
            coreset_pool_subsample=CORESET_POOL_SUBSAMPLE,
        )

    results[acquisition_name] = run_pool_based_active_learning_experiment(
        trainer,
        data,
        acquisition_name=acquisition_name,
        al_iters=AL_ITERS,
        query_batch_size=QUERY_BATCH_SIZE,
        acquisition_batch_size=ACQUISITION_BATCH,
        **kwargs,
    )

for name, r in results.items():
    print(f"{name:>35s}  final test NLL = {r['final_test_nll']:.4f}  "
          f"(train size: {r['state'].train_inputs.shape[0]})")

---
## Step 6 — Comparison Plots

In [ ]:
plot_loss_comparison(results);

### Query concentration near the phase boundary
The red dashed line is the fraction expected under uniform sampling. Bars above the line mean the method over-samples the transition zone; bars below mean it focuses elsewhere.

In [ ]:
plot_sigma_concentration(results, P=P, transition_zone=(0.5, 1.2));

---
## Step 7 — Inspect Individual Runs

Three-panel figure per method:
1. **Left — Learning curve:** test NLL vs round (convergence and stability)
2. **Centre — Labelled points in $(\sigma, \kappa)$:** where the method chose to query
3. **Right — Output marginal:** histogram of true vs MDN-sampled $q_0(T)$ across the test set (blue = true, red = MDN). Good overlap means the MDN learned the distribution well.

In [ ]:
plot_single_run(results["mi_lb"], P=P, title="MI-LB");

In [ ]:
plot_single_run(results["random"], P=P, title="Random");